In [1]:
!pip install pandas rapidfuzz openpyxl


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import List, Dict, Tuple, Any

from typing import Dict, Tuple, List, Any
from difflib import SequenceMatcher
from rapidfuzz import fuzz
import unicodedata
import re


def normalizar_texto(text: str) -> str:
    """
    Normaliza texto para comparación aproximada.

    Ejemplo:
    'Café en grano, molido, instantáneo'
    -> 'cafe grano molido instantaneo'
    """
    if text is None:
        return ""

    text = str(text).lower().strip()

    text = "".join(
        char for char in unicodedata.normalize("NFD", text)
        if unicodedata.category(char) != "Mn"
    )

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text
    """    stopwords = {
        "de", "del", "la", "el", "los", "las", "y", "con",
        "sin", "para", "en", "por", "tipo"
    }

    tokens = [
        token for token in text.split()
        if token not in stopwords
    ]

    return " ".join(tokens)"""


def char_ngrams(text: str, n: int = 3) -> set[str]:
    text = f" {text} "

    if len(text) < n:
        return {text.strip()} if text.strip() else set()

    return {
        text[i:i + n]
        for i in range(len(text) - n + 1)
    }


def jaccard_similarity(
    a: str,
    b: str,
    ngram_size: int = 3
) -> float:
    grams_a = char_ngrams(a, n=ngram_size)
    grams_b = char_ngrams(b, n=ngram_size)

    if not grams_a or not grams_b:
        return 0.0

    return len(grams_a & grams_b) / len(grams_a | grams_b)


def sequence_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


def token_sort_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0

    return fuzz.token_sort_ratio(a, b) / 100


def token_set_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0

    return fuzz.token_set_ratio(a, b) / 100


def partial_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0

    return fuzz.partial_ratio(a, b) / 100


def score_search_match(
    query: str,
    candidate: str,
    method_threshold: float = 0.75,
    ngram_size: int = 3,
    already_normalized: bool = False,
) -> Tuple[float, List[str], Dict[str, float]]:
    """
    Calcula probabilidad de coincidencia entre una búsqueda y un candidato.

    Si already_normalized=True, asume que query y candidate ya vienen normalizados.
    """

    if ngram_size < 2:
        raise ValueError("ngram_size debe ser mayor o igual a 2.")

    if ngram_size > 5:
        raise ValueError("ngram_size no debería ser mayor a 5.")

    query_normalized = query if already_normalized else normalizar_texto(query)
    candidate_normalized = candidate if already_normalized else normalizar_texto(candidate)

    if not query_normalized or not candidate_normalized:
        return 0.0, [], {}

    exact_score = 1.0 if query_normalized == candidate_normalized else 0.0

    contains_score = (
        1.0
        if query_normalized in candidate_normalized
        or candidate_normalized in query_normalized
        else 0.0
    )

    scores = {
        "exact": exact_score,
        "contains": contains_score,
        "sequence": sequence_similarity(query_normalized, candidate_normalized),
        "token_sort": token_sort_similarity(query_normalized, candidate_normalized),
        "token_set": token_set_similarity(query_normalized, candidate_normalized),
        "partial": partial_similarity(query_normalized, candidate_normalized),
        "char_ngram_jaccard": jaccard_similarity(
            query_normalized,
            candidate_normalized,
            ngram_size=ngram_size,
        ),
    }

    probability = sum(scores.values()) / len(scores)

    valid_methods = [
        method
        for method, score in scores.items()
        if score >= method_threshold
    ]

    return (
        round(float(probability), 4),
        valid_methods,
        {
            method: round(float(score), 4)
            for method, score in scores.items()
        }
    )

def buscar_producto_similar(
    new_product_name: str,
    products: List[Any],
    top_n: int = 5,
    min_probability: float = 0.55,
    method_threshold: float = 0.75,
    ngram_size: int = 3,
) -> List[Tuple[Any, float, List[str], Dict[str, float]]]:
    """
    Busca productos similares dentro de una lista de objetos.

    Cada producto debe tener al menos un atributo:
    - name

    Retorna:
    [
        (producto, probability, valid_methods, method_scores),
        ...
    ]
    """

    query_normalized = normalizar_texto(new_product_name)

    if not query_normalized:
        return []

    results = []

    for product in products:
        candidate_name = getattr(product, "name", None)

        if candidate_name is None:
            continue

        candidate_normalized = normalizar_texto(candidate_name)

        probability, valid_methods, method_scores = score_search_match(
            query=query_normalized,
            candidate=candidate_normalized,
            method_threshold=method_threshold,
            ngram_size=ngram_size,
            already_normalized=True,
        )

        if probability >= min_probability and valid_methods:
            results.append(
                (
                    product,
                    probability,
                    valid_methods,
                    method_scores,
                )
            )

    results.sort(key=lambda item: item[1], reverse=True)

    return results[:top_n]

In [7]:
def buscar_producto_similar(
    new_product_name: str,
    products: List[Any],
    top_n: int = 5,
    min_probability: float = 0.55,
    method_threshold: float = 0.75,
    ngram_size: int = 3,
) -> List[Tuple[Any, float, List[str], Dict[str, float]]]:
    """
    Busca productos similares dentro de una lista de objetos.

    Cada producto debe tener al menos un atributo:
    - name

    Retorna:
    [
        (producto, probability, valid_methods, method_scores),
        ...
    ]
    """

    query_normalized = normalizar_texto(new_product_name)

    if not query_normalized:
        return []

    results = []

    for product in products:
        candidate_name = getattr(product, "name", None)

        if candidate_name is None:
            continue

        candidate_normalized = normalizar_texto(candidate_name)

        probability, valid_methods, method_scores = score_search_match(
            query=query_normalized,
            candidate=candidate_normalized,
            method_threshold=method_threshold,
            ngram_size=ngram_size,
            already_normalized=True,
        )

        if probability >= min_probability and valid_methods:
            results.append(
                (
                    product,
                    probability,
                    valid_methods,
                    method_scores,
                )
            )

    results.sort(key=lambda item: item[1], reverse=True)

    return results[:top_n]

In [ ]:
import csv
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import List
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

@dataclass
class CBATimelinePoint:
    original_name: str
    anio: int
    mes: int
    periodo: str
    unidad_medida_original: str
    precio_unidad_medida: float
    precio_100g: float


@dataclass
class CBAProduct:
    category: str
    name: str
    cost_timeline_per100g: list[CBATimelinePoint] = field(default_factory=list)

    def plot_timeline(self):
        puntos = sorted(
            self.cost_timeline_per100g,
            key=lambda p: (p.anio, p.mes)
        )

        fechas = [
            pd.Timestamp(p.anio, p.mes, 1)
            for p in puntos
        ]

        precios = [
            p.precio_100g
            for p in puntos
        ]

        fig, ax = plt.subplots(figsize=(14, 6))

        ax.plot(
            fechas,
            precios,
            marker="o",
            markersize=3
        )

        ax.set_title(
            f"Evolución histórica del precio de {self.name}"
        )
        ax.set_xlabel("Año")
        ax.set_ylabel("Precio por 100 g (Q)")

        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

        ax.grid(True, alpha=0.3)

        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
    
MESES = [
    "01ene", "02feb", "03mar", "04abr",
    "05may", "06jun", "07jul", "08ago",
    "09sep", "10oct", "11nov", "12dic"
]

COLUMNAS = [
    "no",
    "categoria",
    "producto",
    "unidad_medida",
    "gramos_hogar_dia",
    "precio_promedio_simple_por_unidad_medida",
    "costo_diario_hogar_4_77_personas",
]

# gramos por mililitro
DENSIDADES_PRODUCTOS = \
{
    "leche": 1.030

}

def parsear_unidad_medida(columna_unidad_medida: str) -> tuple[float, str]:
    """Convierte: '460 gms', '1000 ml', '750 ml'

    Retorna:
    - cantidad
    - unidad normalizada
    """
    cantidad, unidad = columna_unidad_medida.strip().split(maxsplit=1)

    cantidad = float(cantidad.replace(",", ""))
    unidad = unidad.strip().lower()

    if unidad in {"g", "gm", "gms", "gramo", "gramos"}:
        unidad = "g"
    elif unidad in {"ml", "mililitro", "mililitros"}:
        unidad = "ml"
    else:
        raise ValueError(f"Unidad no reconocida: {columna_unidad_medida}")

    return cantidad, unidad


def calcular_precio_100g(precio: float, unidad_medida: str) -> float:
    cantidad, unidad = parsear_unidad_medida(unidad_medida)

    if unidad == "g":
        return (precio / cantidad) * 100

    if unidad == "ml":
        # Pendiente, buscar densidades de productos
        return None

    raise ValueError(f"Unidad no reconocida: {unidad}")

def search_product(new_product_name: str, products: List[CBAProduct]) -> CBAProduct | None:
    new_name_norm = normalizar_texto(new_product_name)

    for product in products:
        if normalizar_texto(product.name) == new_name_norm:
            return product

    return None


def procesar_cba(cba_mensual: pd.DataFrame, products: list[CBAProduct]) -> None:
    for _, row in cba_mensual.iterrows():
        original_name = str(row["producto"]).strip()
        category = str(row["categoria"]).strip()
        unidad_medida = str(row["unidad_medida"]).strip()
        precio = float(row["precio_promedio_simple_por_unidad_medida"])

        precio_100g = calcular_precio_100g(precio, unidad_medida)

        # Por ahora ignoramos productos medidos en ml.
        if precio_100g is None:
            continue

        product = search_product(original_name, products)

        if product is None:
            product = CBAProduct(
                category=category,
                name=original_name,
            )
            products.append(product)
        
        point = CBATimelinePoint(
            original_name=original_name,
            anio=int(row["anio"]),
            mes=int(row["mes"]),
            periodo=str(row["periodo"]),
            unidad_medida_original=unidad_medida,
            precio_unidad_medida=precio,
            precio_100g=precio_100g,
        )

        product.cost_timeline_per100g.append(point)

def leer_cbas_csv(filepath: Path) -> pd.DataFrame:
    rows = []

    with open(filepath, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)

        header = next(reader, None)

        for line_no, row in enumerate(reader, start=2):
            if not row or all(cell.strip() == "" for cell in row):
                continue

            if len(row) == 7:
                rows.append(row)
                continue

            if len(row) > 7:
                # no, categoria, producto..., unidad_medida, gramos, precio, costo
                no = row[0]
                categoria = row[1]
                producto = ",".join(row[2:-4]).strip()
                unidad_medida = row[-4]
                gramos_hogar_dia = row[-3]
                precio = row[-2]
                costo = row[-1]

                rows.append([
                    no,
                    categoria,
                    producto,
                    unidad_medida,
                    gramos_hogar_dia,
                    precio,
                    costo,
                ])
                continue

            raise ValueError(
                f"Fila incompleta en {filepath.name}, línea {line_no}: {row}"
            )

    df = pd.DataFrame(rows, columns=COLUMNAS)

    df["no"] = pd.to_numeric(df["no"], errors="raise").astype(int)

    df["gramos_hogar_dia"] = (
        df["gramos_hogar_dia"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .pipe(pd.to_numeric, errors="raise")
    )

    df["precio_promedio_simple_por_unidad_medida"] = pd.to_numeric(
        df["precio_promedio_simple_por_unidad_medida"],
        errors="raise"
    )

    df["costo_diario_hogar_4_77_personas"] = pd.to_numeric(
        df["costo_diario_hogar_4_77_personas"],
        errors="raise"
    )

    return df


def obtener_cba(anio: int, mes: int, path="../data/raw/ine/cba_historicos", productos: list[CBAProduct] = []) -> pd.DataFrame | None:
    path = Path(path)

    if mes < 1 or mes > len(MESES):
        raise ValueError(f"Mes inválido, debe ser un número entre 1 y 12 ({mes})")
    if anio > 2023 or anio < 2017:
        return None

    filename = f"cba_{anio}_{MESES[mes - 1]}_tabla.csv"
    filepath = path / filename

    if not filepath.exists():
        return None

    cba_mensual = leer_cbas_csv(filepath)


    cba_mensual["anio"] = anio
    cba_mensual["mes"] = mes
    cba_mensual["periodo"] = f"{anio}_{MESES[mes - 1]}"

    procesar_cba(cba_mensual, productos)
    return cba_mensual


def obtener_siguiente_fecha(anio: int, mes: int) -> tuple[int, int]:
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1


def cargar_cbas_desde(anio_inicio=2017, mes_inicio=10) -> list:
    anio_actual = anio_inicio
    mes_actual = mes_inicio

    cbas = []
    products: list[CBAProduct] = []


    current_cba = obtener_cba(anio_actual,mes_actual,productos=products)
    while current_cba is not None:
        cbas.append(current_cba)

        anio_actual, mes_actual = obtener_siguiente_fecha(
            anio_actual,
            mes_actual
        )

        current_cba = obtener_cba(anio_actual,mes_actual,productos=products)

    return products


productos = cargar_cbas_desde()

print()

In [6]:
productos[0].cost_timeline_per100g

[CBATimelinePoint(original_name='ARROZ', anio=2017, mes=10, periodo='2017_10oct', unidad_medida_original='460 gms', precio_unidad_medida=4.97, precio_100g=1.0804347826086957),
 CBATimelinePoint(original_name='ARROZ', anio=2017, mes=11, periodo='2017_11nov', unidad_medida_original='460 gms', precio_unidad_medida=4.97, precio_100g=1.0804347826086957),
 CBATimelinePoint(original_name='ARROZ', anio=2017, mes=12, periodo='2017_12dic', unidad_medida_original='460 gms', precio_unidad_medida=4.97, precio_100g=1.0804347826086957),
 CBATimelinePoint(original_name='ARROZ', anio=2018, mes=1, periodo='2018_01ene', unidad_medida_original='460 gms', precio_unidad_medida=4.97, precio_100g=1.0804347826086957),
 CBATimelinePoint(original_name='ARROZ', anio=2018, mes=2, periodo='2018_02feb', unidad_medida_original='460 gms', precio_unidad_medida=4.97, precio_100g=1.0804347826086957),
 CBATimelinePoint(original_name='ARROZ', anio=2018, mes=3, periodo='2018_03mar', unidad_medida_original='460 gms', precio_